# Iterative filtering and one current sheet

$J^2$ around the chosen sheet and along the cut through it, with or without iterative filtering of B (`FILTERING`),
and the B and J spectra before and after filtering with the wavenumber where the density SNR drops to 3 (from
`quality_checks/SNR`). A cut can be saved as the reference that later cuts are drawn against.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pta import config
from pta.fields import current_density
from pta.filtering import iterative_filtering3D
from pta.plotting import save_figure, show_field, use_style
from pta.sheets import descent_direction, line_profile
from pta.spectra import summed_spectrum

use_style()
RESULTS = config.results_dir("single_sheet", "IterativeFilteringComparison")
CACHE = config.cache_dir("single_sheet", "IterativeFilteringComparison")

In [ ]:
FILTERING = True
BETA_P = 1
TIME = 250

# per run: iterative filtering (IMFs from index on, Xi), the cell of the chosen sheet, and k d_i where SNR = 3
CASES = {
    0.0625: dict(index=1, Xi=1.6, sheet=(3731, 2294), snr3_k=38.5105),
    1: dict(index=2, Xi=2.0, sheet=(3732, 2228), snr3_k=14.1252),
    16: dict(index=3, Xi=2.0, sheet=(3708, 2284), snr3_k=4.3192),
}
ZOOM_HALF_BOX = 500                   # cells
CUT_HALF_LENGTH = 60
REFERENCE_PROFILE = CACHE / "J2_reference_cut.npz"
SAVE_AS_REFERENCE = False             # True: save this cut as the reference
K_RANGE_SHOWN = (5e-2, 50)
POWER_RANGE_SHOWN = (1e-6, 1)

In [ ]:
case = CASES[BETA_P]
run = config.run("beta_scan", beta_p=BETA_P)
dx, dy = run.dx, run.dy
B = run.load_many(["Bx", "By", "Bz"], TIME)
J = current_density(*B, dx, dy)
if FILTERING:
    B_filtered = iterative_filtering3D(*B, case["index"], Xi=case["Xi"])
    J_filtered = current_density(*B_filtered, dx, dy)
J2 = sum(component**2 for component in (J_filtered if FILTERING else J))
label = f"{'filtered' if FILTERING else 'unfiltered'}_beta_p{BETA_P}"
when = rf"$\beta_p = {BETA_P}$, t = {TIME}" + (", filtered" if FILTERING else "")

In [ ]:
sheet = case["sheet"]
fig, ax = plt.subplots(figsize=(6, 6))
i0, j0 = max(sheet[0] - ZOOM_HALF_BOX, 0), max(sheet[1] - ZOOM_HALF_BOX, 0)
show_field(ax, J2[i0:sheet[0] + ZOOM_HALF_BOX, j0:sheet[1] + ZOOM_HALF_BOX], dx, dy, label=r"$J^2$",
           corner=(i0 * dx, j0 * dy))
ax.scatter([(sheet[0] + 0.5) * dx], [(sheet[1] + 0.5) * dy], color="white", marker="x")
ax.set_title(r"$J^2$, " + when, fontsize=10)
save_figure(fig, RESULTS, f"J2_sheet_{label}")
plt.show()

In [ ]:
direction = descent_direction(J2, sheet)
s, profile = line_profile(J2, sheet, direction, CUT_HALF_LENGTH)
if SAVE_AS_REFERENCE:
    np.savez(REFERENCE_PROFILE, s=s * dx, profile=profile)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(s * dx, profile, label="this cut")
if REFERENCE_PROFILE.exists():
    reference = np.load(REFERENCE_PROFILE)
    ax.plot(reference["s"], reference["profile"], "--", label="reference")
ax.set_xlabel(r"Distance along the cut ($d_p$)")
ax.set_ylabel(r"$J^2$")
ax.set_title(r"$J^2$ along the steepest descent, " + when, fontsize=10)
ax.legend()
save_figure(fig, RESULTS, f"J2_cut_{label}")
plt.show()

In [ ]:
if FILTERING:
    fig, ax = plt.subplots()
    k, spectrum_B = summed_spectrum(B, dx, dy)
    _, spectrum_J = summed_spectrum(J, dx, dy)
    _, spectrum_B_filtered = summed_spectrum(B_filtered, dx, dy)
    _, spectrum_J_filtered = summed_spectrum(J_filtered, dx, dy)
    line_B, = ax.loglog(k, spectrum_B, label="B spectrum")
    line_J, = ax.loglog(k, spectrum_J, label="J spectrum")
    ax.loglog(k, spectrum_J_filtered, linestyle=":", color=line_J.get_color(), label="J spectrum with filter")
    ax.loglog(k, spectrum_B_filtered, linestyle=":", color=line_B.get_color(), label="B spectrum with filter")
    ax.axvline(case["snr3_k"], linestyle="--", color="red", label=r"$k d_i$ at SNR = 3")
    ax.set_xlabel(r"$k$")
    ax.set_ylabel(r"$P(k)$")
    ax.set_xlim(*K_RANGE_SHOWN)
    ax.set_ylim(*POWER_RANGE_SHOWN)
    ax.set_title(when, fontsize=10)
    ax.legend()
    save_figure(fig, RESULTS, f"B_and_J_spectra_beta_p{BETA_P}")
    plt.show()